# 27 RRT*：渐近最优采样


本课按百科条目写：先定义……调度台在最后，可跳过。

---

## 1. 定义

**RRT\***（Karaman & Frazzoli, IJRR 2011）在 RRT 接受 \(x_{\mathrm{new}}\) 之后多两步：

1. **选父（choose parent）：** 在半径 r 的邻域 Near 里，选使 \(g(x)+\|x-x_{\mathrm{new}}\|\) 最小、且连线无碰的节点当父亲。不再固定「几何最近」。
2. **重接（rewire）：** 对 Near 里每个邻居 x，若走 \(x_{\mathrm{new}}\) 再走到 x 更便宜、且线段无碰，则把 x 的父指针改到 \(x_{\mathrm{new}}\)，并更新 \(g(x)\)。

- **输入：** 与 RRT 相同，外加邻域半径 r（本课 **固定** r，便于手算）。
- **输出：** 同样的树 + 各点代价 g；成功时路径代价随样本变多 **几乎必然趋向最优**（渐近最优）。

理论半径随 n 缩小，约 \(\gamma(\log n / n)^{1/d}\)。教学用常数 r，例如库默认 0.14。

---

## 2. 和谁相邻

|  | RRT | RRT\* | A\* |
|--|-----|--------|------|
| 改已有边 | 否，边一旦加上就冻住 | 要 rewire | 图的边本来就在 |
| 最优 | 否 | 渐近（n→∞） | 有限图一次弹出即最优 |
| 邻域 | 只 nearest 一个 | 半径 r 内一批 | 图邻居 |
| 适用 | 先尽快找到一条 | 还想把路径拉直、拉短 | 已有离散图 |

RRT 先碰到 goal 就停（本库如此）。RRT\* 通常继续采样，用更好的 g 刷新到 goal 的路径。本库 `rrt_star` 跑满 n 次，保留 `best_path`。

---

## 3. 不变量 / 定理

- 每个节点的 g 是 **当前树** 上根到该点的折线长。rewire 只在 `cand + 1e-9 < cost[j]` 且线段无碰时改父。
- **渐近最优：** 在合适的 r(n)、缝有正宽度等条件下，n→∞ 时解的代价以概率 1 趋向最优代价。有限 n 只保证「不差于同一前缀树的 RRT 行为」，不保证已经最优。
- **碰撞不变量：** 选父和 rewire **每一条候选边** 都要线段检测。省掉则树会穿障，g 再小也非法。
- 教学局限：本课与库只更新被 rewire 的那个邻居的 g，不沿其子树向下推。节点很少时影响小；严格实现应递减子树。

---

## 4. 完整伪代码（增量，接在 RRT 接受 x_new 之后）

```
# nodes, parent, cost 已有；x_new 尚未入树
Near ← { j | hypot(nodes[j], x_new) <= r }

p ← nearest 的下标
c ← cost[p] + dist(nodes[p], x_new)
for j in Near:
    cand ← cost[j] + dist(nodes[j], x_new)
    if cand < c and not SegCollides(nodes[j], x_new):
        c, p ← cand, j

nodes.append(x_new); parent.append(p); cost.append(c)
ni ← |nodes|-1

for j in Near:
    cand ← c + dist(x_new, nodes[j])
    if cand < cost[j] and not SegCollides(x_new, nodes[j]):
        parent[j] ← ni
        cost[j] ← cand
```

与下面 4 点例子逐行相同；例子里空场，碰撞恒假。

---

## 5. 复杂度

每点扫 Near。若 Near=全体节点，每次 \(O(n)\) 次距离+碰撞，总计 \(O(n^2)\)（再乘线段采样）。半径缩小后 Near 期望更小；再加 KD 树可接近 \(O(n\log n)\)。**不要**在大 n 时把 Near 定义成全部节点还声称这是 RRT\* 的标准复杂度。

---

## 6. 完整手算 / 小表

空场四点（插入前三个、再插入第四个）。半径 r=2.5。

| idx | 点 | parent | g |
|-----|----|--------|---|
| 0 | (0,0) 起点 | None | 0 |
| 1 | (4,0) | 0 | 4 |
| 2 | (2,1) | 1 | 4+√5 ≈ 6.236 |

插入 \(x_{\mathrm{new}}=(0,1)\)：

| j | dist(j, new) | 在 Near？ | 经 j 到 new 的 cand |
|---|--------------|-----------|---------------------|
| 0 | 1 | 是 | 0+1=1 |
| 1 | √17≈4.123 | 否（>2.5） | — |
| 2 | 2 | 是 | 6.236+2=8.236 |

选父：j=0，g(new)=1。new 成为 idx=3。

Rewire：

| j | 旧 g | 经 new 的 cand | 动作 |
|---|------|----------------|------|
| 0 | 0 | 1+1=2 | 不改（更大） |
| 2 | 6.236 | 1+2=**3** | **parent[2]=3，g=3** |

点 2 原来绕道 (4,0)，现在经 (0,1) 更短。这就是 rewire 的全部几何。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 7. 分步实现（自己写，print traces）

把第 6 节的表写成代码：打印插入前后的 parent/cost。空场不碰撞；真实 RRT\* 必须把 `seg_collides` 填回去。


In [ ]:
import math

nodes = [(0.0, 0.0), (4.0, 0.0), (2.0, 1.0)]
parent = [None, 0, 1]
cost = [0.0, 4.0, 4.0 + math.hypot(2.0, 1.0)]
radius = 2.5
x_new = (0.0, 1.0)


def dist(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])


def dump(title):
    print(title)
    for i, p in enumerate(nodes):
        print(f"  idx={i}  p={p}  parent={parent[i]}  g={cost[i]:.3f}")


dump("插入前")
nb = [j for j, q in enumerate(nodes) if dist(q, x_new) <= radius]
print("Near", nb, "半径", radius)

p = min(nb, key=lambda j: cost[j] + dist(nodes[j], x_new))
c = cost[p] + dist(nodes[p], x_new)
print(f"选父 p={p}  g_new={c:.3f}")

nodes.append(x_new)
parent.append(p)
cost.append(c)
ni = len(nodes) - 1

for j in nb:
    cand = c + dist(nodes[j], x_new)
    # 真 RRT* 这里还要 not seg_collides(x_new, nodes[j])
    if cand + 1e-9 < cost[j]:
        print(f"rewire {j}: parent {parent[j]} -> {ni}, g {cost[j]:.3f} -> {cand:.3f}")
        parent[j] = ni
        cost[j] = cand

dump("插入后")
print("点 2 的父应从 1 变成 3，g 从 6.236 变成 3")


## 8. 逐行实现表

| 行 | 作用 |
|----|------|
| `nb = [j if dist<=r]` | Near；本例不含点 1 |
| 选父 `min(nb, key=g+dist)` | 不是几何 nearest（nearest 碰巧也是 0） |
| 先 append 再 rewire | `ni` 必须是 new 的下标 |
| `cand + 1e-9 < cost[j]` | 严格变好才改，避免数值抖动来回接 |
| 注释里的 `seg_collides` | 空场省略；有障省掉就会穿墙抄近路 |
| 不更新点 2 的孩子 | 本例点 2 无孩子；有孩子时应下推 g |

库在 `rrt_star` 里对每个新点做这两段，并在靠近 goal 时刷新 `best_path`。


In [ ]:
from lib.maps import continuous_scene
from lib.algos.rrt import rrt, rrt_star

scene = continuous_scene()
a = rrt(scene, seed=2, n=900)
b = rrt_star(scene, seed=2, n=1400)
print("RRT    found", a["found"], "cost", a["cost"], "nodes", len(a["nodes"]))
print("RRT*   found", b["found"], "cost", b["cost"], "nodes", len(b["nodes"]))
print("RRT* 代价应 <= RRT:", b["cost"] <= a["cost"] + 1e-6)


## 9. 常见 bug

1. **rewire 不做线段碰撞。** g 变小的边可能穿过圆，路径非法。
2. **Near = 全部节点。** 正确但 \(O(n^2)\)；n 到万级会卡。半径 / KD 树是正道。
3. **选父仍只用几何最近。** 那就还是 RRT，没有渐近最优。
4. **rewire 改了 parent 却不改 cost。** 以后选父用过期 g，树会越改越乱。
5. **有限 n 就声称最优。** 只是渐近；同 seed 比较时应有 RRT\* 代价 ≤ RRT（eval `test_27`）。

## 10. 对照库

`rrt_star` 默认 `n=1200, step=0.06, radius=0.14`，goal bias 0.1。选父与 rewire 都调用 `_seg_collides`。同 seed 下再加大 n，代价应非增。


In [ ]:
from lib.studio.widget import PlannerStudio

studio = PlannerStudio()
studio.show_rrt(scene, b, "RRT*")
studio


## 条目小结

| 项目 | 内容 |
|------|------|
| 增量 | 选父（半径内最小 g+dist）+ rewire |
| 保证 | 渐近最优；有限 n 不保证已最优 |
| 半径 | 理论随 n 变；本课固定 |
| 碰撞 | 候选边每条都查线段 |
| 对照 | 同 seed，RRT\* 代价 ≤ RRT |

## 练习

1. 不看代码，根据第 6 节表写出插入后四行 parent/g。点 2 的父是谁？
2. 若 Near 误设为全部节点，本例点 1 会不会被 rewire？cand 是多少？
3. 在圆心与 (0,1)–(2,1) 之间放一个圆，rewire 到点 2 的边可能撞。此时应保持 parent[2]=1。缺碰撞检查会怎样？
